# Desafio 1 · Python, API e dados mockados · Serviço Meteorológico

Neste notebook eu conecto o Python à planilha do Google Sheets pela API, gero uma base fictícia do tema Serviço Meteorológico e envio as duas tabelas para a planilha.

- `estacoes` (dimensão): cadastro das estações, com a chave `estacao_id`. Gerei 10.050 linhas.
- `leituras` (fato): leituras do tempo feitas pelas estações, com a chave `leitura_id` e a chave estrangeira `estacao_id`. Gerei 62.000 linhas.

Todos os dados são fictícios, feitos só para treino.

## Parte 1 · Conexão e teste

### Célula 01 · Instalar as bibliotecas gspread e faker

In [ ]:
%pip install -q gspread faker

### Célula 02 · Importações e autenticação (service account)

A credencial fica guardada na Secret `GCP_SA_JSON` do Colab, assim ela não aparece no código.

In [2]:
# Bibliotecas usadas ao longo do notebook.
import json, gspread, random, os, time, math
from faker import Faker
import pandas as pd

# date, datetime e timedelta são usados para criar e somar datas.
from datetime import date, datetime, timedelta

# No Colab, as credenciais ficam nas Secrets. Fora do Colab (ex.: VS Code),
# o código usa os arquivos credenciais.json (credencial) e gs_id.txt (ID da planilha).
try:
    from google.colab import files, userdata
    EM_COLAB = True
except ModuleNotFoundError:
    EM_COLAB = False

def ler_segredo(nome):
    # Devolve o valor de um segredo, buscando no lugar certo para cada ambiente.
    if EM_COLAB:
        return userdata.get(nome)
    arquivo = "credenciais.json" if nome == "GCP_SA_JSON" else "gs_id.txt"
    with open(arquivo, encoding="utf-8") as f:
        return f.read().strip()

# Lê o conteúdo do JSON da credencial (Secret GCP_SA_JSON) e converte o texto em dicionário.
info = json.loads(ler_segredo("GCP_SA_JSON"))

# Cria o cliente do gspread autenticado com a service account.
gc = gspread.service_account_from_dict(info)

# Exibe o e-mail da credencial para confirmar que a autenticação funcionou.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@projeto-meteorologia.iam.gserviceaccount.com


### Célula 03 · Abrir a planilha

O ID da planilha também fica numa Secret (`GS_ID`), para não ficar exposto no repositório.

In [3]:
# Lê o ID da planilha guardado na Secret GS_ID.
SHEET_ID = ler_segredo("GS_ID")

# Nome da tabela dimensão do tema: será o nome da primeira aba da planilha.
NOME_TABELA = "estacoes"

# Abre a planilha pelo ID e seleciona a primeira aba (ws será usada nas próximas células).
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# Renomeia a aba com o nome da tabela e exibe os dados da planilha.
ws.update_title(NOME_TABELA)
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Meteorologia | estacoes | https://docs.google.com/spreadsheets/d/18k9DNEyGZx9jOPZrFWdlp8RO3asxBdvR3fJDLduNgKo


### Célula 04 · Cabeçalho · Tabela Dimensão (`estacoes`)

In [4]:
# Nomes EXATOS das colunas da tabela dimensão, na mesma ordem da documentação do tema.
COLUNAS_DIMENSAO = ["estacao_id", "nome_estacao", "cidade", "pais",
                    "latitude", "longitude", "altitude_m"]

# Escreve o cabeçalho a partir da célula A1.
ws.update(range_name="A1", values=[COLUNAS_DIMENSAO])

# Lê a linha 1 de volta e compara com COLUNAS_DIMENSAO.
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS_DIMENSAO)

Cabeçalho confere: True


### Célula 05 · Inserir 10 linhas de teste · Tabela Dimensão

Montei a função `gerar_linha(i)` seguindo a documentação do tema:

- o país é sorteado primeiro, com os pesos da documentação (Brasil 30%, Estados Unidos 10%, e assim por diante);
- a cidade depende do país: no Brasil uso o Faker e nos outros países escolho uma entre 5 cidades;
- latitude e longitude são sorteadas dentro de uma área aproximada de cada país;
- a altitude é um número inteiro entre 0 e 3.499 metros.

Atenção: essa célula só deve ser executada uma vez, senão as 10 linhas entram de novo na planilha.

In [5]:
# Gerador do Faker em português do Brasil (usado nas cidades brasileiras e nas datas).
fake = Faker("pt_BR")

# Sementes fixas: a cada execução os valores aleatórios saem iguais (resultado reproduzível).
Faker.seed(42)
random.seed(42)

# Países e pesos definidos na documentação do tema (a soma dá 100).
PAISES = ["Brasil", "Estados Unidos", "Argentina", "Canadá", "Alemanha", "França",
          "Portugal", "Japão", "Austrália", "México", "África do Sul", "Índia",
          "Reino Unido", "Itália", "Chile"]
PESOS_PAISES = [30, 10, 6, 6, 6, 6, 5, 5, 5, 5, 4, 4, 3, 3, 2]

# Caixa geográfica aproximada de cada país: (latitude mínima, latitude máxima, longitude mínima, longitude máxima).
CAIXA_PAIS = {
    "Brasil":         (-33.0,   5.0, -73.0, -35.0),
    "Estados Unidos": ( 25.0,  49.0, -124.0, -67.0),
    "Argentina":      (-55.0, -22.0, -73.0, -54.0),
    "Canadá":         ( 42.0,  70.0, -140.0, -53.0),
    "Alemanha":       ( 47.3,  55.0,   6.0,  15.0),
    "França":         ( 42.5,  51.0,  -4.5,   8.0),
    "Portugal":       ( 37.0,  42.0,  -9.5,  -6.2),
    "Japão":          ( 31.0,  45.0, 129.0, 145.0),
    "Austrália":      (-39.0, -11.0, 113.0, 153.0),
    "México":         ( 15.0,  32.0, -117.0, -87.0),
    "África do Sul":  (-34.8, -22.0,  16.5,  32.9),
    "Índia":          (  8.0,  35.0,  68.0,  97.0),
    "Reino Unido":    ( 50.0,  58.6,  -8.0,   1.8),
    "Itália":         ( 37.0,  47.0,   6.6,  18.5),
    "Chile":          (-55.0, -17.5, -75.6, -67.0),
}

# Lista de 5 cidades por país (o Brasil usa nomes gerados pelo Faker).
CIDADES_PAIS = {
    "Estados Unidos": ["Nova York", "Chicago", "Denver", "Seattle", "Miami"],
    "Argentina":      ["Buenos Aires", "Córdoba", "Rosário", "Mendoza", "Salta"],
    "Canadá":         ["Toronto", "Montreal", "Vancouver", "Calgary", "Ottawa"],
    "Alemanha":       ["Berlim", "Munique", "Hamburgo", "Colônia", "Frankfurt"],
    "França":         ["Paris", "Lyon", "Marselha", "Toulouse", "Bordéus"],
    "Portugal":       ["Lisboa", "Porto", "Coimbra", "Braga", "Faro"],
    "Japão":          ["Tóquio", "Osaka", "Sapporo", "Fukuoka", "Nagoya"],
    "Austrália":      ["Sydney", "Melbourne", "Brisbane", "Perth", "Adelaide"],
    "México":         ["Cidade do México", "Guadalajara", "Monterrey", "Puebla", "Mérida"],
    "África do Sul":  ["Joanesburgo", "Cidade do Cabo", "Durban", "Pretória", "Porto Elizabeth"],
    "Índia":          ["Nova Délhi", "Mumbai", "Bangalore", "Chennai", "Calcutá"],
    "Reino Unido":    ["Londres", "Manchester", "Edimburgo", "Birmingham", "Glasgow"],
    "Itália":         ["Roma", "Milão", "Nápoles", "Turim", "Florença"],
    "Chile":          ["Santiago", "Valparaíso", "Concepción", "Antofagasta", "Punta Arenas"],
}

def gerar_linha(i):
    # Devolve UM registro fictício de estação, no formato {nome_da_coluna: valor}.
    # "i" é o número da linha e serve de identificador sequencial.

    # Coluna de origem: o país é sorteado primeiro e define cidade e coordenadas.
    pais = random.choices(PAISES, weights=PESOS_PAISES)[0]
    lat_min, lat_max, lon_min, lon_max = CAIXA_PAIS[pais]
    cidade = fake.city() if pais == "Brasil" else random.choice(CIDADES_PAIS[pais])

    return {
        "estacao_id": i,                                           # identificador sequencial (PK)
        "nome_estacao": f"EST-{i:05d}",                            # código no formato EST-00001
        "cidade": cidade,                                          # cidade coerente com o país
        "pais": pais,                                              # país sorteado com os pesos da documentação
        "latitude": round(random.uniform(lat_min, lat_max), 4),    # dentro da caixa do país
        "longitude": round(random.uniform(lon_min, lon_max), 4),   # dentro da caixa do país
        "altitude_m": random.randint(0, 3499),                     # altitude em metros (0 a 3.499)
    }

# Gera 10 registros de teste.
linhas = [gerar_linha(i) for i in range(1, 11)]
print(linhas[0])

# Converte cada registro em lista de valores NA ORDEM do cabeçalho e envia para a planilha.
valores = [[linha[c] for c in COLUNAS_DIMENSAO] for linha in linhas]
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas")

{'estacao_id': 1, 'nome_estacao': 'EST-00001', 'cidade': 'Paris', 'pais': 'França', 'latitude': 48.8032, 'longitude': -1.4389, 'altitude_m': 571}
10 linhas enviadas


### Célula 06 · Conferência · Tabela Dimensão

In [6]:
# Lê de volta tudo o que está na aba para conferir o que realmente chegou à planilha.
valores_lidos = ws.get_all_values()

print("Linhas (cabeçalho + dados):", len(valores_lidos))    # esperado: 11
print("Cabeçalho confere:", valores_lidos[0] == COLUNAS_DIMENSAO)
print(valores_lidos[:3])

Linhas (cabeçalho + dados): 10061
Cabeçalho confere: True
[['estacao_id', 'nome_estacao', 'cidade', 'pais', 'latitude', 'longitude', 'altitude_m'], ['1', 'EST-00001', 'Braga', 'Portugal', '37,8338', '-8,4127', '590'], ['2', 'EST-00002', 'Córdoba', 'Argentina', '-38,677', '-59,1101', '2109']]


## Parte 2 · Base completa

### Célula 07 · Esquema · Tabela Fato (`leituras`)

In [7]:
# Nome da tabela fato do tema: será o nome da segunda aba da planilha.
NOME_TABELA_FATO = "leituras"

# Nomes EXATOS das colunas da tabela fato, na mesma ordem da documentação do tema.
COLUNAS_FATO = ["leitura_id", "estacao_id", "data_hora", "condicao_tempo",
                "temperatura_c", "umidade_pct", "precipitacao_mm",
                "velocidade_vento_kmh", "pressao_hpa"]

# Coluna da tabela fato que é a chave estrangeira (guarda o ID da tabela dimensão).
COLUNA_FK = "estacao_id"

# Confere se a chave estrangeira é mesmo uma das colunas da tabela fato.
assert COLUNA_FK in COLUNAS_FATO, "COLUNA_FK precisa ser um dos nomes de COLUNAS_FATO"
print(NOME_TABELA_FATO, "|", COLUNAS_FATO)

leituras | ['leitura_id', 'estacao_id', 'data_hora', 'condicao_tempo', 'temperatura_c', 'umidade_pct', 'precipitacao_mm', 'velocidade_vento_kmh', 'pressao_hpa']


### Célula 08 · Estrutura · Tabela Fato

A função `gerar_linha_fato(i, ids_dim, info_dim)` gera uma leitura. A condição do tempo é sorteada primeiro e muda as outras colunas, como está na documentação:

| Condição (peso) | Temperatura | Umidade (média · desvio) | Vento (gama, forma 2, escala) | Pressão | Chuva |
| --- | --- | --- | --- | --- | --- |
| `ensolarado` (38%) | +1,5 °C | 45 % · 12 | 5 (média 10 km/h) | +1 hPa | nenhuma |
| `nublado` (50%) | 0 °C | 65 % · 12 | 6 (média 12 km/h) | 0 hPa | nenhuma |
| `chuvoso` (9%) | −2,0 °C | 85 % · 7 | 7 (média 14 km/h) | −3 hPa | exponencial, média 4 mm (mínimo 0,1 mm) |
| `tempestade` (3%) | −4,0 °C | 92 % · 5 | 14 (média 28 km/h) | −8 hPa | 5 mm + exponencial de média 7 mm |

- Na chave estrangeira, as primeiras leituras passam por todas as estações da lista `ids_dim`, assim toda estação fica com pelo menos uma leitura. Depois disso a estação é sorteada com `random.choice(ids_dim)`.
- A temperatura depende da latitude, da altitude, da época do ano (invertida entre os hemisférios norte e sul), da hora do dia e da condição do tempo, com um pouco de variação aleatória.
- A pressão cai conforme a altitude aumenta, e a umidade fica sempre entre 20 e 100.
- Precisei passar o `info_dim`, que guarda a latitude e a altitude de cada estação, porque a temperatura e a pressão dependem delas.

In [8]:
# Condições do tempo e pesos definidos na documentação do tema.
CONDICOES = ["nublado", "ensolarado", "chuvoso", "tempestade"]
PESOS_CONDICOES = [50, 38, 9, 3]

# Efeito de cada condição sobre as demais variáveis (tabela acima).
AJUSTE_CONDICAO = {
    #             temperatura, umidade média, umidade desvio, escala do vento, pressão
    "ensolarado": (+1.5, 45, 12,  5, +1),
    "nublado":    ( 0.0, 65, 12,  6,  0),
    "chuvoso":    (-2.0, 85,  7,  7, -3),
    "tempestade": (-4.0, 92,  5, 14, -8),
}

# Período simulado das leituras: 01/01/2025 a 31/07/2026.
INICIO_LEITURAS = datetime(2025, 1, 1)
FIM_LEITURAS = datetime(2026, 7, 31, 23, 59, 59)

def gerar_linha_fato(i, ids_dim, info_dim):
    # Devolve UMA leitura fictícia, no formato {nome_da_coluna: valor}.
    # "i" é o número da linha; "ids_dim" é a lista de IDs que existem na tabela dimensão;
    # "info_dim" guarda (latitude, altitude_m) de cada estação.

    # --- Chave estrangeira: primeiro uma leitura para cada estação, depois sorteio ---
    estacao_id = ids_dim[i - 1] if i <= len(ids_dim) else random.choice(ids_dim)
    latitude, altitude = info_dim[estacao_id]

    # --- Valores de origem: o momento da leitura e a condição do tempo ---
    momento = fake.date_time_between(start_date=INICIO_LEITURAS, end_date=FIM_LEITURAS)
    condicao = random.choices(CONDICOES, weights=PESOS_CONDICOES)[0]
    aj_temp, umid_media, umid_desvio, escala_vento, aj_pressao = AJUSTE_CONDICAO[condicao]

    # --- Temperatura: latitude, altitude, época do ano, hora do dia e condição ---
    dia_do_ano = momento.timetuple().tm_yday
    sazonal = 6 * math.cos(2 * math.pi * (dia_do_ano - 196) / 365.25)   # pico em meados de julho no hemisfério norte
    if latitude < 0:
        sazonal = -sazonal                                              # estações invertidas no hemisfério sul
    hora = momento.hour + momento.minute / 60
    diaria = 5 * math.cos(2 * math.pi * (hora - 15) / 24)               # máxima por volta das 15 h
    temperatura = (25 - abs(latitude) / 3 - 0.0065 * altitude
                   + sazonal + diaria + aj_temp + random.gauss(0, 2))

    # --- Umidade, vento e pressão dependem da condição ---
    umidade = min(100.0, max(20.0, random.gauss(umid_media, umid_desvio)))
    vento = random.gammavariate(2, escala_vento)
    pressao = random.gauss(1013, 8) - 0.10 * altitude + aj_pressao

    # --- Chuva: só existe em leituras chuvoso e tempestade ---
    if condicao == "chuvoso":
        chuva = max(0.1, round(random.expovariate(1 / 4), 1))
    elif condicao == "tempestade":
        chuva = round(5 + random.expovariate(1 / 7), 1)
    else:
        chuva = 0.0

    return {
        "leitura_id": i,                                        # identificador (renumerado em ordem cronológica na Célula 09)
        "estacao_id": estacao_id,                               # CHAVE ESTRANGEIRA: um ID que existe na dimensão
        "data_hora": momento.strftime("%Y-%m-%d %H:%M:%S"),     # data e hora da leitura
        "condicao_tempo": condicao,                             # ensolarado, nublado, chuvoso ou tempestade
        "temperatura_c": round(temperatura, 1),                 # graus Celsius
        "umidade_pct": round(umidade, 1),                       # % entre 20 e 100
        "precipitacao_mm": chuva,                               # milímetros (0 quando não chove)
        "velocidade_vento_kmh": round(vento, 1),                # km/h
        "pressao_hpa": round(pressao, 1),                       # hPa
    }

# Teste com as 10 estações de teste da Célula 05.
info_teste = {l["estacao_id"]: (l["latitude"], l["altitude_m"]) for l in linhas}
print(gerar_linha_fato(1, list(info_teste), info_teste))

{'leitura_id': 1, 'estacao_id': 1, 'data_hora': '2025-03-21 14:38:54', 'condicao_tempo': 'ensolarado', 'temperatura_c': 9.9, 'umidade_pct': 20.0, 'precipitacao_mm': 0.0, 'velocidade_vento_kmh': 27.9, 'pressao_hpa': 950.5}


### Célula 09 · Gerar as tabelas

Aqui gero as duas tabelas: 10.050 estações e 62.000 leituras. No final ordenei as leituras pela `data_hora` e numerei o `leitura_id` de novo, porque a documentação pede que ele siga a ordem das datas.

In [9]:
# Quantidade de linhas de cada tabela (volumetria da documentação). O total precisa ser de pelo menos 20.000.
N_DIM = 10_050
N_FATO = 62_000

# Reinicia as sementes: o resultado fica reproduzível.
Faker.seed(42)
random.seed(42)

# ---- Tabela dimensão: usa a função gerar_linha (Célula 05) ----
tabela_dim = pd.DataFrame([gerar_linha(i) for i in range(1, N_DIM + 1)], columns=COLUNAS_DIMENSAO)

# ---- Tabela fato: usa a função gerar_linha_fato (Célula 08) ----
# ids_dim é a lista de IDs da dimensão GERADA acima: é nela que a fato busca a chave estrangeira.
ids_dim = list(tabela_dim[COLUNAS_DIMENSAO[0]])
info_dim = dict(zip(tabela_dim["estacao_id"], zip(tabela_dim["latitude"], tabela_dim["altitude_m"])))
tabela_fato = pd.DataFrame([gerar_linha_fato(i, ids_dim, info_dim) for i in range(1, N_FATO + 1)], columns=COLUNAS_FATO)

# Ordena as leituras pelo momento e renumera o leitura_id (1, 2, 3...) em ordem cronológica.
tabela_fato = tabela_fato.sort_values("data_hora", kind="stable").reset_index(drop=True)
tabela_fato["leitura_id"] = range(1, len(tabela_fato) + 1)

# Guarda as tabelas usando o nome de cada uma.
BASES_GERADAS = {NOME_TABELA: tabela_dim, NOME_TABELA_FATO: tabela_fato}

print({nome: len(df) for nome, df in BASES_GERADAS.items()}, "| total:", len(tabela_dim) + len(tabela_fato))
display(tabela_dim.head())
display(tabela_fato.head())

{'estacoes': 10050, 'leituras': 62000} | total: 72050


,estacao_id,nome_estacao,cidade,pais,latitude,longitude,altitude_m
0,1,EST-00001,Paris,França,48.8032,-1.4389,571
1,2,EST-00002,Nagoya,Japão,32.2171,135.7507,122
2,3,EST-00003,Alves,Brasil,-24.1589,-50.1233,2298
3,4,EST-00004,da Cruz de Goiás,Brasil,-8.3044,-52.2922,902
4,5,EST-00005,Rosário,Argentina,-28.2888,-72.8765,3300


,leitura_id,estacao_id,data_hora,condicao_tempo,temperatura_c,umidade_pct,precipitacao_mm,velocidade_vento_kmh,pressao_hpa
0,1,7381,2025-01-01 00:10:36,tempestade,0.6,96.9,15.4,25.2,703.4
1,2,5565,2025-01-01 00:39:56,nublado,2.7,56.0,0.0,12.1,710.9
2,3,5017,2025-01-01 00:56:50,nublado,7.2,50.0,0.0,31.9,738.5
3,4,2415,2025-01-01 00:57:37,ensolarado,2.2,56.3,0.0,26.0,735.4
4,5,9443,2025-01-01 01:05:06,tempestade,-17.9,95.5,19.2,33.9,744.5


### Célula 10 · Validar e gerar os arquivos CSV

Antes de gravar os CSV, essa célula confere as regras do guia e as regras da documentação do tema. Se alguma não for atendida, a execução para e mostra qual foi o problema.

In [10]:
# ---- Regras do guia ----
assert list(tabela_dim.columns) == COLUNAS_DIMENSAO, "Colunas diferentes do esquema na tabela dimensão"
assert list(tabela_fato.columns) == COLUNAS_FATO, "Colunas diferentes do esquema na tabela fato"
assert tabela_dim[COLUNAS_DIMENSAO[0]].is_unique, "ID repetido na tabela dimensão"
assert tabela_fato["leitura_id"].is_unique, "ID repetido na tabela fato"
assert tabela_fato[COLUNA_FK].isin(tabela_dim[COLUNAS_DIMENSAO[0]]).all(), "Chave estrangeira sem ID na dimensão"
assert len(tabela_dim) + len(tabela_fato) >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# ---- Regras da documentação do tema ----
assert tabela_dim["estacao_id"].isin(tabela_fato["estacao_id"]).all(), "Há estação sem nenhuma leitura"
assert tabela_fato["data_hora"].is_monotonic_increasing, "Leituras fora de ordem cronológica"
lat_brasil = tabela_dim.loc[tabela_dim["pais"] == "Brasil", "latitude"]
assert lat_brasil.between(-33, 5).all(), "Latitude do Brasil fora da faixa (-33 a 5)"
assert tabela_fato["umidade_pct"].between(0, 100).all(), "Umidade fora da faixa (0 a 100)"
assert set(tabela_fato["condicao_tempo"]) <= set(CONDICOES), "Condição do tempo fora dos 4 valores válidos"
com_chuva = tabela_fato["condicao_tempo"].isin(["chuvoso", "tempestade"])
assert (tabela_fato.loc[com_chuva, "precipitacao_mm"] > 0).all(), "Leitura chuvosa ou de tempestade sem chuva"
assert (tabela_fato.loc[~com_chuva, "precipitacao_mm"] == 0).all(), "Chuva em leitura ensolarada ou nublada"
medias = tabela_fato.groupby("condicao_tempo")[["pressao_hpa", "umidade_pct", "velocidade_vento_kmh"]].mean()
assert medias.loc["tempestade", "pressao_hpa"] < medias.loc["ensolarado", "pressao_hpa"], "Tempestade deveria ter pressão menor"
assert medias.loc["tempestade", "umidade_pct"] > medias.loc["ensolarado", "umidade_pct"], "Tempestade deveria ter umidade maior"
assert medias.loc["tempestade", "velocidade_vento_kmh"] > medias.loc["ensolarado", "velocidade_vento_kmh"], "Tempestade deveria ter vento maior"
print("Todas as validações passaram.")
display(medias.round(1))

# ---- Grava um CSV por tabela, lê de volta para conferir e baixa o arquivo (só no Colab) ----
os.makedirs("bases_geradas", exist_ok=True)
for nome, df in BASES_GERADAS.items():
    caminho = f"bases_geradas/{nome}.csv"
    df.to_csv(caminho, index=False, encoding="utf-8")
    lido = pd.read_csv(caminho)
    assert len(lido) == len(df) and list(lido.columns) == list(df.columns), f"CSV diferente em {nome}"
    print(nome + ".csv ->", len(lido), "linhas")
    if EM_COLAB:
        files.download(caminho)

Todas as validações passaram.


,pressao_hpa,umidade_pct,velocidade_vento_kmh
condicao_tempo,,,
chuvoso,836.6,84.9,14.1
ensolarado,839.8,45.1,10.0
nublado,837.0,65.0,12.0
tempestade,828.8,91.7,28.2


estacoes.csv -> 10050 linhas
leituras.csv -> 62000 linhas


### Célula 11 · Função para o envio de dados para a planilha

A função `enviar` manda a tabela para a planilha em lotes de 5.000 linhas, com uma pausa de 2 segundos entre um lote e outro, porque a API tem limite de escritas por minuto. Se a aba já existe, ela é limpa e usada de novo; se não existe, é criada. No final ela confere se a quantidade de linhas na planilha está certa.

In [11]:
def enviar(sh, df, nome_aba, lote=5000, pausa=2):
    # Formata o número com ponto como separador de milhar (ex.: 62.000).
    n = lambda x: f"{x:,}".replace(",", ".")

    # Reaproveita a aba se ela já existir (limpando o conteúdo) ou cria uma nova.
    try:
        aba = sh.worksheet(nome_aba)
        aba.clear()
    except gspread.WorksheetNotFound:
        aba = sh.add_worksheet(title=nome_aba, rows=len(df) + 1, cols=len(df.columns))

    # Ajusta o tamanho da aba e grava o cabeçalho na linha 1.
    aba.resize(rows=len(df) + 1, cols=len(df.columns))
    aba.update(range_name="A1", values=[list(df.columns)])

    # Converte a tabela em lista de linhas (valor ausente vira texto vazio).
    valores = df.astype(object).where(df.notna(), "").values.tolist()

    # Envia em lotes, com pausa para respeitar o limite de escritas por minuto.
    for i in range(0, len(valores), lote):
        aba.append_rows(valores[i:i + lote], value_input_option="RAW")
        print(f"  {nome_aba}: {n(min(i + lote, len(valores)))} / {n(len(valores))}")
        time.sleep(pausa)

    # Lê de volta e confere a quantidade de linhas (sem contar o cabeçalho).
    lidas = len(aba.col_values(1)) - 1
    if lidas != len(df):
        raise RuntimeError(f"Aba {nome_aba}: enviei {len(df)}, li {lidas}.")
    print(f"OK {nome_aba}: {n(len(df))} linhas na planilha")

### Célula 12 · Enviar as tabelas geradas

Essa célula substitui as 10 linhas de teste da Célula 05, porque a função limpa a aba antes de enviar. A aba `leituras` é criada aqui. Dá para rodar de novo sem duplicar as linhas.

In [12]:
# Envia cada tabela gerada para a aba de mesmo nome (dimensão e fato).
for nome, df in BASES_GERADAS.items():
    enviar(sh, df, nome)

  estacoes: 5.000 / 10.050
  estacoes: 10.000 / 10.050
  estacoes: 10.050 / 10.050
OK estacoes: 10.050 linhas na planilha
  leituras: 5.000 / 62.000
  leituras: 10.000 / 62.000
  leituras: 15.000 / 62.000
  leituras: 20.000 / 62.000
  leituras: 25.000 / 62.000
  leituras: 30.000 / 62.000
  leituras: 35.000 / 62.000
  leituras: 40.000 / 62.000
  leituras: 45.000 / 62.000
  leituras: 50.000 / 62.000
  leituras: 55.000 / 62.000
  leituras: 60.000 / 62.000
  leituras: 62.000 / 62.000
OK leituras: 62.000 linhas na planilha


### Célula 13 · Validar a base na planilha

In [13]:
# Soma e exibe a quantidade total de linhas das tabelas.
total = sum(len(df) for df in BASES_GERADAS.values())
print("Total de linhas:", f"{total:,}".replace(",", "."))
assert total >= 20_000, "Abaixo do mínimo (20.000 linhas somadas)."

# Confere os cabeçalhos e a quantidade de registros em cada aba.
for nome, df in BASES_GERADAS.items():
    w = sh.worksheet(nome)
    assert w.row_values(1) == list(df.columns), f"Cabeçalho diferente em {nome}"
    assert len(w.col_values(1)) - 1 == len(df), f"Contagem diferente em {nome}"

# Confere o relacionamento: toda chave estrangeira da tabela fato existe na tabela dimensão.
fk_fato = BASES_GERADAS[NOME_TABELA_FATO][COLUNA_FK]
ids_dim = BASES_GERADAS[NOME_TABELA][COLUNAS_DIMENSAO[0]]
assert fk_fato.isin(ids_dim).all(), "Há chave estrangeira sem ID correspondente na tabela dimensão"

print("A planilha confere com a base gerada.")
print("Abas da planilha:", [w.title for w in sh.worksheets()])

Total de linhas: 72.050
A planilha confere com a base gerada.
Abas da planilha: ['estacoes', 'leituras']
